In [1]:
import os
from dotenv import load_dotenv

env = "LOCAL"
if env == "GPU":
    os.environ["HF_HOME"] = rf"/cs/student/project_msc/2025/dsml/navnmann/huggingface"
    load_dotenv(rf"/cs/student/project_msc/2025/dsml/navnmann/MSc_code/.env")

In [ ]:
import ast, copy, json, random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.preprocessing import normalize
from sklearn.metrics import (
    average_precision_score,
    adjusted_rand_score,
    precision_recall_fscore_support,
)

from sentence_transformers import SentenceTransformer
import networkx as nx
from pyvis.network import Network

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


### Helper Constant and Functions

In [ ]:
SEED = 72
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("../data")
SRC_CSV = DATA_DIR / "combined_2017_2023_themes_with_KG.csv"
SYNTH_CSV = DATA_DIR / "synthetic_golden_set.csv"
GOLD_CSV = DATA_DIR / "synthetic_gold_themes.csv"
SPLIT_CSV = DATA_DIR / "synthetic_gold_split.csv"
THEME_EMB_CACHE = DATA_DIR / "theme_bge_m3.npz"
ENT_EMB_CACHE = DATA_DIR / "entity_bge_m3.npz"


RESULTS_DIR = Path("../output/exp3_chapter6_rgcn")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
HUB_MAX_THEMES = 200  # drop too common entities

# Data split ratios
TRAIN_RATIO = 0.6
VAL_RATIO = 0.2
TEST_RATIO = 0.2

# model parameters
DEVICE = "cpu"
HIDDEN = 256
OUT_DIM = 128
NUM_BASES = None  # since 12 is a small number; no need for this!
DROPOUT = 0.4
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 300
EVAL_EVERY = 10
TEMP = 0.2
TAU_GRID = [round(x, 3) for x in np.arange(0.02, 0.30, 0.02)]

THEME_RELS = ["HAS_ACTOR", "HAS_TARGET", "HAS_CONTEXT", "AFFECTS"]
ENT_RELS = ["ACTS_ON", "BELONGS_TO"]
BASE_RELS = THEME_RELS + ENT_RELS
print("Device:", DEVICE)

device cpu


In [ ]:
def safe_parse(value):

    if isinstance(value, list):
        return value
    try:
        return json.loads(value)
    except Exception:

        try:
            return ast.literal_eval(value)
        except Exception:
            return []


def build_split(df):

    # shuffle and split clusters
    cluster_ids = np.array(sorted(df["gold_cluster_id"].unique()))
    rng = np.random.default_rng(SEED)
    rng.shuffle(cluster_ids)

    n_train = int(TRAIN_RATIO * len(cluster_ids))
    n_val = int(VAL_RATIO * len(cluster_ids))

    split_of_cluster = {}
    for cluster in cluster_ids[:n_train]:
        split_of_cluster[cluster] = "train"

    for cluster in cluster_ids[n_train : n_train + n_val]:
        split_of_cluster[cluster] = "val"

    for cluster in cluster_ids[n_train + n_val :]:
        split_of_cluster[cluster] = "test"

    # save split to CSV
    table = df[["theme", "gold_cluster_id"]].copy()
    table["split"] = table["gold_cluster_id"].map(split_of_cluster)
    table.to_csv(SPLIT_CSV, index=False)

    return table


def load_bge_cache(cache_path, names):

    vector_of = {}
    if cache_path.exists():
        cache = np.load(cache_path, allow_pickle=True)
        vector_of = dict(zip(cache["themes"].tolist(), cache["vecs"]))

    missing = [name for name in names if name not in vector_of]
    print(f"{len(names) - len(missing):,} cached, missing - {len(missing):,}")

    if missing:

        encoder = SentenceTransformer("BAAI/bge-m3")
        new_vectors = encoder.encode(
            missing, batch_size=64, show_progress_bar=True, normalize_embeddings=True
        )
        vector_of.update(zip(missing, new_vectors.astype(np.float32)))
        # rewrite cache; old + new vectors
        np.savez_compressed(
            cache_path,
            themes=np.array(list(vector_of.keys()), dtype=object),
            vecs=np.stack(list(vector_of.values())).astype(np.float32),
        )

    matrix = np.stack([vector_of[name] for name in names]).astype(np.float32)
    matrix = np.nan_to_num(matrix, nan=0.0, posinf=0.0, neginf=0.0)

    return normalize(matrix, norm="l2", axis=1)


def supervised_contrastive_loss(
    train_node_ids, train_cluster_labels, all_node_embeddings
):
    """push themes within the same cluster closer! this is suited for canonicalization task"""

    train_node_embeddings = F.normalize(
        all_node_embeddings[train_node_ids.to(DEVICE)], dim=1
    )

    # temp sharpens distributions (more peaked softmax for smaller value)
    pair_wise_similarity = (train_node_embeddings @ train_node_embeddings.t()) / TEMP
    n_train = train_node_embeddings.shape[0]

    # theme must ignore itself
    self_mask = torch.eye(n_train, dtype=torch.bool, device=DEVICE)
    labels = train_cluster_labels.to(DEVICE)

    is_same_cluster = (labels[:, None] == labels[None, :]) & ~self_mask
    pair_wise_similarity = pair_wise_similarity.masked_fill(
        self_mask, -1e9
    )  # exclude from softmax

    # softmax over each row (non partner acts as negatives)
    # partner are positives
    log_prob = pair_wise_similarity - torch.logsumexp(
        pair_wise_similarity, dim=1, keepdim=True
    )

    # average log prob over each themes partners (ignore loss for partnerless themes)
    n_partners = is_same_cluster.sum(1)
    has_partner_mask = n_partners > 0

    # sum each partner's loss
    per_theme_loss = -(log_prob * is_same_cluster).sum(1)[has_partner_mask]

    # avg per theme loss
    return (per_theme_loss / n_partners[has_partner_mask]).mean()

### Preprocessing

In [5]:
real_articles = pd.read_csv(SRC_CSV)
real_articles = real_articles[
    real_articles["themes"].notna() & real_articles["knowledge_graph"].notna()
]
synthetic_articles = pd.read_csv(SYNTH_CSV)

# combine real and synthetic articles
all_kg_strings = pd.concat(
    [real_articles["knowledge_graph"], synthetic_articles["knowledge_graph"]],
    ignore_index=True,
)
df = pd.DataFrame({"kg": all_kg_strings.apply(safe_parse)})
print(f"{len(real_articles)} real + {len(synthetic_articles)} synthetic articles")


# each KG record is a triplet of type (head, head_type, relation, tail, tail_type) tagged with its theme
# for every theme we gather the set of entities (the heads/tails) that appear alongside it.
entities_of_theme = defaultdict(set)
for article_kg in df["kg"]:

    for record in article_kg:

        # extract triplet and theme
        triplet = record.get("triplet", [])
        theme = record.get("theme", "")
        if len(triplet) != 5 or not theme:
            continue

        head, head_type, relation, tail, tail_type = triplet

        for entity, entity_type in ((head, head_type), (tail, tail_type)):
            # filter entity types + skip theme's own CONCEPT node
            if entity != theme:  # and entity_type in KEEP_ENTITY_TYPES:
                entities_of_theme[theme].add(entity)

# count how many distinct themes each entity touches
themes_per_entity = Counter()
for entity_set in entities_of_theme.values():
    for entity in entity_set:
        themes_per_entity[entity] += 1
hub_entities = {e for e, count in themes_per_entity.items() if count > HUB_MAX_THEMES}


# drop hub entities + keep themes that still have at least one entity
kept_entities_of_theme = {}
for theme, entity_set in entities_of_theme.items():
    kept = entity_set - hub_entities
    if kept:
        kept_entities_of_theme[theme] = kept


# name -> integer-id lookup for themes and entities
# theme ids are used in the RGCN node embeddings, and the entity ids are used in the edge list
theme_list = sorted(kept_entities_of_theme)
entity_list = sorted({e for ents in kept_entities_of_theme.values() for e in ents})
theme_id = {theme: idx for idx, theme in enumerate(theme_list)}
entity_id = {entity: idx for idx, entity in enumerate(entity_list)}
num_themes, num_entities = len(theme_list), len(entity_list)


pair_rows = []
for theme in theme_list:
    n_entities = len(kept_entities_of_theme[theme])
    pair_rows.extend([theme_id[theme]] * n_entities)
pair_rows = np.array(pair_rows)

print(
    f"Themes {num_themes:,}; Entities {num_entities:,}; Hub entities dropped {len(hub_entities)}"
)

5433 real + 1068 synthetic articles
Themes 15,296; Entities 34,672; Hub entities dropped 8


In [ ]:
# print and see how it looks!

# theme_list
# entity_list
# theme_id
# entity_id

In [ ]:
gold = pd.read_csv(GOLD_CSV)
# display(gold.head(5))

# filter to retain valid themes and clusters
gold = gold[gold["keep"] == 1]
gold = gold[gold["theme"].isin(theme_id)]
cluster_sizes = gold["gold_cluster_id"].value_counts()
clusters_with_a_pair = cluster_sizes[cluster_sizes >= 2].index
gold = gold[gold["gold_cluster_id"].isin(clusters_with_a_pair)]

# reuse saved split (if it still matches the current gold themes)
# otherwise rebuild it
if SPLIT_CSV.exists():
    split = pd.read_csv(SPLIT_CSV)
    split_is_stale = set(split["theme"]) != set(gold["theme"])

    if split_is_stale:
        print("saved split no longer matches the gold set - rebuilding")
        split = build_split(gold)
else:
    split = build_split(gold)

In [8]:
for split_name in ["train", "val", "test"]:

    split_rows = split[split["split"] == split_name]
    n_in_pool = split_rows["theme"].isin(theme_id).sum()

    print(
        f"{split_name}: {split_rows['gold_cluster_id'].nunique()} clusters, {len(split_rows)} themes"
    )

train: 600 clusters, 1854 themes
val: 200 clusters, 648 themes
test: 201 clusters, 628 themes


In [9]:
theme_vectors = load_bge_cache(THEME_EMB_CACHE, theme_list)
entity_vectors = load_bge_cache(ENT_EMB_CACHE, entity_list)

# entities are stacked below the themes, so the first num_themes rows are the themes
# (num_themes + num_entities, 1024)
node_features = np.vstack([theme_vectors, entity_vectors]).astype(np.float32)

num_nodes = node_features.shape[0]
print(f"number of nodes: {num_nodes}")

15,296 cached, missing - 0
34,672 cached, missing - 0
number of nodes: 49968


In [10]:
relation_to_id = {rel: idx for idx, rel in enumerate(BASE_RELS)}
edges_of_relation = {idx: set() for idx in range(len(BASE_RELS))}

# All edges for this relation
# {0: {(src_id, dst_id),(src_id, dst_id)}
# ...
# 6: {(src_id, dst_id),(src_id, dst_id)}
# }

for article_kg in df["kg"]:

    for record in article_kg:

        triplet = record.get("triplet", [])
        theme = record.get("theme", "")

        if len(triplet) != 5 or not theme:
            continue

        head, head_type, relation, tail, tail_type = triplet

        relation = str(relation).strip().upper().replace(" ", "_")
        if relation not in relation_to_id:
            continue

        if relation in THEME_RELS:
            # theme (CONCEPT) -> entity edge
            theme_node = theme_id.get(theme)
            entity_node = entity_id.get(tail)
            if theme_node is not None and entity_node is not None:
                edges_of_relation[relation_to_id[relation]].add(
                    (theme_node, num_themes + entity_node)
                )
        else:
            # entity -> entity edge
            head_node = entity_id.get(head)
            tail_node = entity_id.get(tail)
            if (
                head_node is not None
                and tail_node is not None
                and head_node != tail_node
            ):
                edges_of_relation[relation_to_id[relation]].add(
                    (num_themes + head_node, num_themes + tail_node)
                )

In [1]:
# print and see how it looks!
# relation_to_id, edges_of_relation

In [ ]:
# each edge is reversed with separate relation_id so message flow both ways
num_base_relations = len(BASE_RELS)
sources, targets, relation_ids = [], [], []
for relation_index, edge_set in edges_of_relation.items():

    for source, target in edge_set:

        # forward
        sources.append(source)
        targets.append(target)
        relation_ids.append(relation_index)

        # inverse
        sources.append(target)
        targets.append(source)
        relation_ids.append(relation_index + num_base_relations)

NUM_REL = 2 * num_base_relations

# standard way to represent RGCN input
x = torch.tensor(node_features)
edge_index = torch.tensor([sources, targets], dtype=torch.long)
edge_type = torch.tensor(relation_ids, dtype=torch.long)

print(
    f"nodes {num_nodes:,}; edges {edge_index.shape[1]:,}; relations {NUM_REL} (including reverse)"
)

for rel in BASE_RELS:
    print(f"{rel:12s}: {len(edges_of_relation[relation_to_id[rel]]):,}")

nodes 49,968; edges 173,190; relations 12 (including reverse)
HAS_ACTOR   : 16,219
HAS_TARGET  : 8,033
HAS_CONTEXT : 17,615
AFFECTS     : 16,351
ACTS_ON     : 19,979
BELONGS_TO  : 8,398


### Model

In [13]:
class RGCN(nn.Module):

    def __init__(self, in_dim, hid, out, num_rel, num_bases, dropout):
        super().__init__()

        self.proj = nn.Linear(in_dim, hid)

        self.conv1 = RGCNConv(hid, hid, num_rel, num_bases=num_bases)
        self.conv2 = RGCNConv(hid, out, num_rel, num_bases=num_bases)

        self.norm1 = nn.LayerNorm(hid)
        self.norm2 = nn.LayerNorm(out)

        # residual: map the BGE-M3 prior (hid) into the output space (out) so it can be added
        self.res = nn.Linear(hid, out)

        self.dropout = dropout

    def forward(self, x, edge_index, edge_type):

        h0 = F.relu(self.proj(x))  # 1024 -> hid, BGE-M3 prior
        h = F.dropout(h0, self.dropout, self.training)

        h = F.relu(self.norm1(self.conv1(h, edge_index, edge_type)))  # hid -> hid
        h = F.dropout(h, self.dropout, self.training)

        h = self.norm2(self.conv2(h, edge_index, edge_type))  # hid -> out

        return h + self.res(h0)

In [ ]:
model = RGCN(x.shape[1], HIDDEN, OUT_DIM, NUM_REL, NUM_BASES, DROPOUT).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
x, edge_index, edge_type = x.to(DEVICE), edge_index.to(DEVICE), edge_type.to(DEVICE)

# much higher (~ 3x) than our RGAT model
print(sum(p.numel() for p in model.parameters()), "parameters to train!")

1574400 parameters to train!


### Train

In [16]:
train_rows = split[(split["split"] == "train") & split["theme"].isin(theme_id)]

# filter for all node_ids for each train theme;
train_node_ids = torch.tensor(
    [theme_id[theme] for theme in train_rows["theme"]], dtype=torch.long
)

# convert gold cluster id "gt_0000" into integers
train_cluster_codes = pd.factorize(train_rows["gold_cluster_id"].values)[0]
train_cluster_labels = torch.tensor(train_cluster_codes, dtype=torch.long)

In [ ]:
val_rows = split[(split.split == "val") & split.theme.isin(theme_id)]
val_node_ids = np.array([theme_id[theme] for theme in val_rows.theme])
val_cluster_codes = pd.factorize(val_rows.gold_cluster_id.values)[0]

pair_i, pair_j = np.triu_indices(len(val_node_ids), k=1)
pair_is_same_cluster = (val_cluster_codes[pair_i] == val_cluster_codes[pair_j]).astype(
    int
)


def val_ap(theme_embeddings):

    vectors = normalize(theme_embeddings[val_node_ids], axis=1)
    pair_cosine = (vectors[pair_i] * vectors[pair_j]).sum(1)

    if not pair_is_same_cluster.sum():
        return float("nan")

    return average_precision_score(pair_is_same_cluster, pair_cosine)

In [ ]:
best = {"ap": -1.0, "state": None, "epoch": -1}

for epoch in range(1, EPOCHS + 1):

    model.train()

    opt.zero_grad()
    loss = supervised_contrastive_loss(
        train_node_ids, train_cluster_labels, model(x, edge_index, edge_type)
    )
    loss.backward()
    opt.step()

    if epoch == 1 or epoch % EVAL_EVERY == 0:

        model.eval()
        with torch.no_grad():
            theme_embeddings = (
                model(x, edge_index, edge_type)[:num_themes].cpu().numpy()
            )
        val_score = val_ap(theme_embeddings)

        if val_score > best["ap"]:
            best = {
                "ap": val_score,
                "state": copy.deepcopy(model.state_dict()),
                "epoch": epoch,
            }

        is_new_best = best["epoch"] == epoch
        print(f"Epoch {epoch:3d} ; loss {loss.item():.4f} ; validation AP {val_score}")


print(f"Best epoch {best['epoch']} ; val AP {best['ap']}")

if best["epoch"] == EPOCHS:
    print(
        f"[warn] best epoch == EPOCHS ({EPOCHS}) - val AP still improving, raise EPOCHS"
    )

Epoch   1 ; loss 5.4543 ; validation AP 0.6172206582185475
Epoch  10 ; loss 4.3894 ; validation AP 0.5472726839828039
Epoch  20 ; loss 4.1321 ; validation AP 0.5769533938951974
Epoch  30 ; loss 3.9546 ; validation AP 0.6253412143811943
Epoch  40 ; loss 3.8234 ; validation AP 0.6580751824038061
Epoch  50 ; loss 3.7253 ; validation AP 0.6859427359307152
Epoch  60 ; loss 3.6375 ; validation AP 0.7059766338310749
Epoch  70 ; loss 3.5530 ; validation AP 0.716072410721687
Epoch  80 ; loss 3.4953 ; validation AP 0.7222286858889228
Epoch  90 ; loss 3.4447 ; validation AP 0.7293646222210197
Epoch 100 ; loss 3.3976 ; validation AP 0.7302784267692033
Epoch 110 ; loss 3.3607 ; validation AP 0.7342434565604371
Epoch 120 ; loss 3.3275 ; validation AP 0.7333699840349793
Epoch 130 ; loss 3.3067 ; validation AP 0.7348739538322047
Epoch 140 ; loss 3.2741 ; validation AP 0.7403588185982896
Epoch 150 ; loss 3.2451 ; validation AP 0.7402281217813326
Epoch 160 ; loss 3.2352 ; validation AP 0.737679655021285

In [19]:
# load best model
model.load_state_dict(best["state"])
model.eval()

# inference on all themes
with torch.no_grad():
    rgcn_emb = (
        model(x, edge_index, edge_type)[:num_themes].cpu().numpy().astype(np.float32)
    )
rgcn_emb = normalize(rgcn_emb, axis=1)

# cache embeddings
OUT_NPZ = RESULTS_DIR / "rgcn_theme_emb.npz"
np.savez(OUT_NPZ, themes=np.array(theme_list, dtype=object), rgcn=rgcn_emb)
print("saved at", OUT_NPZ.name, rgcn_emb.shape)

saved at rgcn_theme_emb.npz (15296, 128)


In [ ]:
def evaluate_split(t2l, split_df, split_name):

    split_rows = split_df[split_df["split"] == split_name]
    split_rows = split_rows[split_rows["theme"].isin(t2l)]

    if len(split_rows) < 2:
        return {"n": len(split_rows)}

    gold_cluster = split_rows["gold_cluster_id"].to_numpy()
    pred_cluster = np.array([t2l[t] for t in split_rows["theme"]])

    i, j = np.triu_indices(len(split_rows), k=1)
    y_true = gold_cluster[i] == gold_cluster[j]
    y_pred = pred_cluster[i] == pred_cluster[j]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return {
        "n": len(split_rows),
        "pair_P": precision,
        "pair_R": recall,
        "pair_F1": f1,
        "ARI": adjusted_rand_score(gold_cluster, pred_cluster),
    }


def tune_and_report(t2l_by_config):

    val_rows = split[split["split"] == "val"]

    best_config, best_val_score = None, -1.0
    for config, t2l in t2l_by_config.items():

        if not np.isnan(val_score) and val_score > best_val_score:
            best_val_score, best_config = val_score, config

    if best_config is None:
        best_config = list(t2l_by_config)[len(t2l_by_config) // 2]

    test = evaluate_split(t2l_by_config[best_config], split, "test")
    row = {
        "tau*": best_config,
        "val_b3_F1": best_val_score,
        "pair_P": test.get("pair_P"),
        "pair_R": test.get("pair_R"),
        "pair_F1": test.get("pair_F1"),
        "ARI": test.get("ARI"),
        "n": test.get("n"),
    }
    return row, t2l_by_config[best_config]

### 2 stage clustering

In [ ]:
MIN_ENTITIES = 3  # min entities entity rich theme
SIMILARITY_THRESHOLDS = [0.5, 0.6, 0.7, 0.8]

# Split themes by how many KG entities they have
n_entities_per_theme = np.bincount(pair_rows, minlength=num_themes)
rich_idx_mask = np.where(n_entities_per_theme >= MIN_ENTITIES)[0]  # graph heavy
poor_idx_mask = np.where(n_entities_per_theme < MIN_ENTITIES)[0]  # phrase heavy
print(f"stage 1 graph-heavy: {len(rich_idx_mask)} themes")
print(f"stage 2 phrase-heavy: {len(poor_idx_mask)} themes")

rich_emb = rgcn_emb[rich_idx_mask]

# pairwise cosine distance = 1 - cosine similarity (embeddings are already L2-normalized)
rich_sim = rich_emb @ rich_emb.T
rich_dist = 1.0 - rich_sim
rich_dist = np.clip(rich_dist, 0.0, None)  # clip tiny negative round-off up to 0
rich_dist = np.nan_to_num(
    rich_dist, nan=1.0
)  # zero-vector themes -> max distance (1.0)
np.fill_diagonal(rich_dist, 0.0)  # ignore self-similarity


Z_rich = linkage(squareform(rich_dist, checks=False), method="complete")
del rich_dist


poor_emb = rgcn_emb[poor_idx_mask]
t2l_by_cfg = {}
attached_by_cfg = {}
for tau in TAU_GRID:
    # cut the stage 1 dendrogram at distance tau -> a cluster label (1..k) per rich theme
    rich_labels = fcluster(Z_rich, t=tau, criterion="distance")
    n_clusters = rich_labels.max()

    # centroid of each stage 1 cluster
    centroids = np.zeros((n_clusters, rgcn_emb.shape[1]), np.float32)
    np.add.at(centroids, rich_labels - 1, rgcn_emb[rich_idx_mask])
    centroids = normalize(centroids, norm="l2", axis=1)

    # for each poor theme, find its nearest stage 1 centroid + that similarity
    if len(poor_idx_mask):
        poor_to_centroid_cos = poor_emb @ centroids.T
        nearest_cluster = np.argmax(poor_to_centroid_cos, axis=1)
        nearest_cos = poor_to_centroid_cos[
            np.arange(len(poor_idx_mask)), nearest_cluster
        ]

    for threshold in SIMILARITY_THRESHOLDS:
        # start every config from the stage 1 assignments
        t2l = {theme_list[i]: f"c{rich_labels[r]}" for r, i in enumerate(rich_idx_mask)}

        attached = 0
        if len(poor_idx_mask):
            # attach a poor theme only if it's similar enough to its nearest centroid
            attach_mask = nearest_cos >= threshold
            attached = int(attach_mask.sum())
            for pi, i in enumerate(poor_idx_mask):
                if attach_mask[pi]:
                    t2l[theme_list[i]] = f"c{nearest_cluster[pi] + 1}"
                else:
                    t2l[theme_list[i]] = f"poor_{i}"  # left as its own singleton

        t2l_by_cfg[(tau, threshold)] = t2l
        attached_by_cfg[(tau, threshold)] = attached


r_2s, t2l_2s = tune_and_report(t2l_by_cfg)
r_2s["attached"] = attached_by_cfg[r_2s["tau*"]]
print(
    f"best (tau, tau_attach) = {r_2s['tau*']} | "
    f"attached {r_2s['attached']}/{len(poor_idx_mask)} entity-poor themes"
)

# a tau* sitting at either end of TAU_GRID means the grid is in the wrong range
chosen_tau = r_2s["tau*"][0]
if chosen_tau in (min(TAU_GRID), max(TAU_GRID)):
    print(f"[warn] tau* {chosen_tau} is at a TAU_GRID boundary - extend the grid")

# res_df = pd.DataFrame([r_2s])
# res_df

stage 1 graph-heavy: 12847 themes
stage 2 phrase-heavy: 2449 themes


/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_14608/2475205813.py:14: RuntimeWarning: divide by zero encountered in matmul
  rich_sim = rich_emb @ rich_emb.T
/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_14608/2475205813.py:14: RuntimeWarning: overflow encountered in matmul
  rich_sim = rich_emb @ rich_emb.T
/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_14608/2475205813.py:14: RuntimeWarning: invalid value encountered in matmul
  rich_sim = rich_emb @ rich_emb.T
/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_14608/2475205813.py:50: RuntimeWarning: divide by zero encountered in matmul
  poor_to_centroid_cos = poor_emb @ centroids.T
/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_14608/2475205813.py:50: RuntimeWarning: overflow encountered in matmul
  poor_to_centroid_cos = poor_emb @ centroids.T
/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_14608/2475205813.py:50: RuntimeWarning: invalid value encountered in matm

best (tau, tau_attach) = (np.float64(0.16), 0.8) | attached 843/2449 entity-poor themes


### Evaluation on the golden set

In [ ]:
# the val-tuned two-stage clustering
final_t2l = t2l_2s

print("HELD OUT TEST SPLIT")
print(
    f"Pairwise P/R/F1: {r_2s['pair_P']:.3f} / {r_2s['pair_R']:.3f} / {r_2s['pair_F1']:.3f}"
)
print(f"ARI: {r_2s['ARI']:.3f}   (n = {r_2s['n']} themes)")


gold = pd.read_csv(GOLD_CSV)
gold = gold[gold["keep"] == 1]
n_gold_total = len(gold)

gold = gold[gold["theme"].isin(final_t2l)].reset_index(drop=True)

# for all themes get gold cluster_id & predicted cluster_id
themes = gold["theme"].to_numpy()
gold_cluster = gold["gold_cluster_id"].to_numpy()
pred_cluster = np.array([final_t2l[t] for t in themes])

i, j = np.triu_indices(len(gold), k=1)
y_true = gold_cluster[i] == gold_cluster[j]
y_pred = pred_cluster[i] == pred_cluster[j]

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary", zero_division=0
)
ari = adjusted_rand_score(gold_cluster, pred_cluster)

train_themes = set(split.loc[split["split"] == "train", "theme"])
n_seen = gold["theme"].isin(train_themes).sum()

print(
    f"WHOLE GOLD SET ({len(gold)} themes, {gold['gold_cluster_id'].nunique()} clusters)"
)
print(f"Positive pair rate: {100 * y_true.mean():.2f}% of {len(y_true):,} pairs")
print(f"Pairwise P/R/F1: {precision:.3f} / {recall:.3f} / {f1:.3f}")
print(f"ARI: {ari:.3f}")

HELD-OUT TEST SPLIT
  pairwise P/R/F1: 0.986 / 0.597 / 0.744
  B-cubed  P/R/F1: 0.902 / 0.780 / 0.836
  ARI: 0.743   (n = 628 themes)

WHOLE GOLD SET  (3130 themes, 1001 clusters)
  [!] 1854 of these themes are in the train split - number is optimistic
  positive pair rate: 0.07% of 4,896,885 pairs
  pairwise P/R/F1: 0.818 / 0.779 / 0.798
  B-cubed  P/R/F1: 0.889 / 0.884 / 0.887
  ARI: 0.798


### Persist the theme -> cluster assignments

In [ ]:
assignments = pd.DataFrame(
    {"theme": list(final_t2l), "canon_cluster": list(final_t2l.values())}
).sort_values("theme")
assignments_path = RESULTS_DIR / "canon_rgcn.csv"
assignments.to_csv(assignments_path, index=False)
print(
    f"wrote {len(assignments)} theme->cluster assignments "
    f"({assignments['canon_cluster'].nunique()} clusters) to {assignments_path}"
)

wrote 15296 theme->cluster assignments (11890 clusters) to ../output/exp3_chapter6_rgcn/canon_rgcn.csv


In [ ]:
theme_to_cluster = final_t2l
SOURCE_CSV = SRC_CSV

REPORT_DIR = RESULTS_DIR / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

JOIN = " | "
REPORT_COLS = [
    "article_id",
    "title",
    "url",
    "date_publish",
    "themes",
    "knowledge_graph",
]


def joined(values):
    """`JOIN`-join, neutralising any literal pipe inside a value so the separator
    stays unambiguous - 312 article titles in this corpus contain one."""
    return JOIN.join(str(v).replace("|", "/") for v in values)


article_rows, triple_rows = [], []
for chunk in pd.read_csv(SOURCE_CSV, usecols=REPORT_COLS, chunksize=20_000):
    chunk = chunk[chunk["themes"].notna() & chunk["knowledge_graph"].notna()]
    for row in chunk.itertuples(index=False):
        for entry in json.loads(row.themes):
            article_rows.append(
                (
                    entry["theme"],
                    entry.get("dimension"),
                    row.article_id,
                    row.title,
                    row.date_publish,
                    row.url,
                )
            )
        for record in json.loads(row.knowledge_graph):
            triplet = record.get("triplet", [])
            if len(triplet) != 5 or not record.get("theme"):
                continue
            head, head_type, relation, tail, tail_type = triplet
            triple_rows.append(
                (
                    record["theme"],
                    row.article_id,
                    head,
                    head_type,
                    relation,
                    tail,
                    tail_type,
                )
            )

theme_articles = pd.DataFrame(
    article_rows,
    columns=["theme", "dimension", "article_id", "title", "date_publish", "url"],
).drop_duplicates()
theme_triples = pd.DataFrame(
    triple_rows,
    columns=[
        "theme",
        "article_id",
        "head",
        "head_type",
        "relation",
        "tail",
        "tail_type",
    ],
)
print(
    f"corpus index: {len(theme_articles):,} theme-article pairs over "
    f"{theme_articles['theme'].nunique():,} themes | {len(theme_triples):,} triples"
)

_archive = np.load(DATA_DIR / "theme_bge_m3.npz", allow_pickle=True)
phrase_vectors = dict(zip(_archive["themes"].tolist(), _archive["vecs"]))

report_members = defaultdict(list)
for theme, cluster in theme_to_cluster.items():
    report_members[cluster].append(theme)

cluster_label, cluster_cohesion = {}, {}
for cluster, members in report_members.items():
    known = [m for m in members if m in phrase_vectors]
    if not known:
        cluster_label[cluster] = sorted(members)[0]
        cluster_cohesion[cluster] = float("nan")
        continue
    vectors = np.stack([phrase_vectors[m] for m in known]).astype(np.float32)
    vectors /= np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)
    centroid = vectors.mean(axis=0)
    centroid /= max(float(np.linalg.norm(centroid)), 1e-12)
    cluster_label[cluster] = known[int(np.argmax(vectors @ centroid))]
    if len(known) > 1:
        sims = vectors @ vectors.T
        cluster_cohesion[cluster] = float(sims[np.triu_indices(len(known), 1)].mean())
    else:
        cluster_cohesion[cluster] = float("nan")

assignments_long = pd.DataFrame(
    {
        "canon_cluster": list(theme_to_cluster.values()),
        "theme": list(theme_to_cluster),
    }
)
assignments_long["cluster_label"] = assignments_long["canon_cluster"].map(cluster_label)


article_map = assignments_long.merge(theme_articles, on="theme", how="left")
article_map["is_synthetic"] = article_map["article_id"].isna()
article_map = article_map[
    [
        "canon_cluster",
        "cluster_label",
        "theme",
        "dimension",
        "article_id",
        "title",
        "date_publish",
        "url",
        "is_synthetic",
    ]
].sort_values(["canon_cluster", "theme", "article_id"], na_position="last")
article_map.to_csv(REPORT_DIR / "cluster_theme_article_map.csv", index=False)

triple_map = assignments_long.merge(theme_triples, on="theme", how="inner")
triple_map = triple_map[
    [
        "canon_cluster",
        "cluster_label",
        "theme",
        "article_id",
        "head",
        "head_type",
        "relation",
        "tail",
        "tail_type",
    ]
].sort_values(["canon_cluster", "theme", "article_id"])
triple_map.to_csv(REPORT_DIR / "cluster_kg_triples.csv", index=False)

themes_with_articles = set(theme_articles["theme"])
real_rows = article_map[~article_map["is_synthetic"]]
articles_by_cluster = dict(tuple(real_rows.groupby("canon_cluster")))
triples_by_cluster = dict(tuple(triple_map.groupby("canon_cluster")))

summary_rows = []
for cluster, members in report_members.items():
    members = sorted(members)
    cluster_articles = articles_by_cluster.get(cluster)
    cluster_triples = triples_by_cluster.get(cluster)

    if cluster_articles is None:
        article_ids, titles, dimensions = [], [], Counter()
    else:
        article_ids = list(dict.fromkeys(cluster_articles["article_id"]))
        dimensions = Counter(cluster_articles["dimension"].dropna())

        ranked = cluster_articles.assign(
            _off_label=cluster_articles["theme"].ne(cluster_label[cluster]),
            _date=pd.to_datetime(cluster_articles["date_publish"], errors="coerce"),
        ).sort_values(["_off_label", "_date"], ascending=[True, False])
        titles = list(dict.fromkeys(ranked["title"].dropna()))

    entities = (
        Counter()
        if cluster_triples is None
        else Counter(zip(cluster_triples["tail"], cluster_triples["tail_type"]))
    )

    summary_rows.append(
        {
            "canon_cluster": cluster,
            "label": cluster_label[cluster],
            "n_themes": len(members),
            "n_articles": len(article_ids),
            "n_triples": 0 if cluster_triples is None else len(cluster_triples),
            "cohesion": cluster_cohesion[cluster],
            "dimensions": joined(f"{d}:{n}" for d, n in dimensions.most_common()),
            "member_themes": joined(members),
            "top_entities": joined(
                f"{entity} ({etype}) ×{n}"
                for (entity, etype), n in entities.most_common(10)
            ),
            "article_ids": joined(article_ids),
            "sample_titles": joined(titles[:5]),
            "n_synthetic_themes": sum(
                1 for m in members if m not in themes_with_articles
            ),
        }
    )

summary = pd.DataFrame(summary_rows).sort_values(
    ["n_themes", "canon_cluster"], ascending=[False, True]
)
summary.to_csv(REPORT_DIR / "cluster_summary.csv", index=False)

covered = sum(1 for t in theme_to_cluster if t in themes_with_articles)
print(
    f"report -> {REPORT_DIR}\n"
    f"cluster_summary.csv            {len(summary):,} clusters\n"
    f"cluster_theme_article_map.csv  {len(article_map):,} rows\n"
    f"cluster_kg_triples.csv         {len(triple_map):,} rows\n"
    f"{covered:,} / {len(theme_to_cluster):,} themes trace to a real article "
    f"({len(theme_to_cluster) - covered:,} synthetic)"
)

corpus index: 17,201 theme-article pairs over 13,268 themes | 87,969 triples


/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_14608/1492125124.py:119: RuntimeWarning: divide by zero encountered in matmul
  cluster_label[cluster] = known[int(np.argmax(vectors @ centroid))]
/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_14608/1492125124.py:119: RuntimeWarning: overflow encountered in matmul
  cluster_label[cluster] = known[int(np.argmax(vectors @ centroid))]
/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_14608/1492125124.py:119: RuntimeWarning: invalid value encountered in matmul
  cluster_label[cluster] = known[int(np.argmax(vectors @ centroid))]
/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_14608/1492125124.py:121: RuntimeWarning: divide by zero encountered in matmul
  sims = vectors @ vectors.T
/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_14608/1492125124.py:121: RuntimeWarning: overflow encountered in matmul
  sims = vectors @ vectors.T
/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_1460

report -> ../output/exp3_chapter6_rgcn/report
  cluster_summary.csv            11,890 clusters
  cluster_theme_article_map.csv  19,229 rows
  cluster_kg_triples.csv         87,957 rows
  13,260 / 15,296 themes trace to a real article (2,036 synthetic)


In [ ]:
originals = gold[gold["role"] == "original"]
orig_themes = originals["theme"].to_numpy()
orig_gold = originals["gold_cluster_id"].to_numpy()
orig_pred = np.array([final_t2l[t] for t in orig_themes])

rows_v = {}
for k in ("syn1", "syn2", "syn3"):
    variants = gold[gold[f"in_{k}"]]
    var_themes = variants["theme"].to_numpy()
    var_gold = variants["gold_cluster_id"].to_numpy()
    var_pred = np.array([final_t2l[t] for t in var_themes])

    y_true = []
    y_pred = []
    for i in range(len(originals)):
        for j in range(len(variants)):

            # skip identical phrases (trivially in the same cluster)
            if orig_themes[i] == var_themes[j]:
                continue
            y_true.append(orig_gold[i] == var_gold[j])
            y_pred.append(orig_pred[i] == var_pred[j])
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    rows_v[f"parent-{k}"] = {
        "n_pairs": y_true.size,
        "n_pos": int(y_true.sum()),
        "precision": p,
        "recall": r,
        "f1": f,
    }

print(pd.DataFrame(rows_v).T.round(3).to_string())

               n_pairs   n_pos  precision  recall     f1
parent-syn1  1078229.0  1096.0      0.869   0.809  0.838
parent-syn2  1093586.0  1132.0      0.865   0.786  0.824
parent-syn3  1090517.0  1129.0      0.864   0.787  0.823


### Inspect Clusters

In [26]:
clusters = defaultdict(list)
for t, lab in final_t2l.items():
    clusters[lab].append(t)

cluster_sizes = sorted((len(v) for v in clusters.values()), reverse=True)
print(f"{len(clusters)} clusters over {num_themes:,} themes")
print(f"Largest 10 cluster sizes: {cluster_sizes[:10]}")
print(f"Singleton clusters (size = 1): {cluster_sizes.count(1)}")

11890 clusters over 15,296 themes
Largest 10 cluster sizes: [32, 25, 24, 21, 16, 16, 15, 15, 14, 14]
Singleton clusters (size = 1): 10216


In [ ]:
G = nx.MultiDiGraph()
for kg_list in df["kg"]:
    for rec in kg_list:
        triplet = rec.get("triplet", [])
        theme = rec.get("theme", "")
        if len(triplet) != 5 or not theme:
            continue
        h, h_type, rel, t, t_type = triplet
        G.add_node(h, entity_type=h_type, is_theme=(str(h_type).upper() == "CONCEPT"))
        G.add_node(t, entity_type=t_type, is_theme=(str(t_type).upper() == "CONCEPT"))
        G.add_edge(h, t, relation=rel, theme=theme)

print(f"Nodes: {G.number_of_nodes():,}, Edges: {G.number_of_edges():,}")

Nodes: 48,976, Edges: 104,173


In [28]:
COLOR_MAP = {
    "COMP": "#f28e2b",
    "ORG": "#b07aa1",
    "ORG/GOV": "#ff9da7",
    "ORG/REG": "#fabfd2",
    "REG": "#fabfd2",
    "GPE": "#bab0ac",
    "PERSON": "#9c755f",
    "EVENT": "#edc948",
    "PRODUCT": "#d37295",
    "SECTOR": "#e15759",
    "CONCEPT": "#FF6347",  # theme anchors
    "ECON_INDICATOR": "#76b7b2",
    "MACRO_TREND": "#e377c2",
    "TECHNOLOGY": "#86bcb6",
    "TECHNOLOGY_CONCEPT": "#86bcb6",
    "POLICY": "#8cd17d",
    "POLICY_OR_REGULATION": "#8cd17d",
    "FIN_INSTRUMENT": "#59a14f",
    "CRYPTO": "#4e79a7",
}


def plot_kg(subG, output=None):
    output = output or RESULTS_DIR / "kg_view.html"
    net = Network(
        height="750px",
        width="100%",
        directed=True,
        notebook=True,
        cdn_resources="in_line",
    )
    net.force_atlas_2based()
    for node, data in subG.nodes(data=True):
        etype = data.get("entity_type", "UNKNOWN")
        is_theme = data.get("is_theme", False)
        net.add_node(
            node,
            label=node,
            color=COLOR_MAP.get(etype, "#cccccc"),
            title=f"<b>{node}</b><br>Type: {etype}"
            + ("<br><i>theme anchor</i>" if is_theme else ""),
            size=30 if is_theme else 18,
            shape="diamond" if is_theme else "dot",
        )
    for u, v, data in subG.edges(data=True):
        net.add_edge(
            u,
            v,
            label=data.get("relation", ""),
            title=f"Rel: {data.get('relation','')}<br>Theme: {data.get('theme','')}",
            arrows="to",
            color="#555555",
        )
    net.show_buttons(filter_=["physics"])
    net.show(output)
    print(f"Saved -> {output}")

In [29]:
def viz_cluster_kg(cluster_label, output_dir=RESULTS_DIR):
    """Combined KG of every theme in a Leiden cluster (CONCEPT theme hubs + entities)."""

    cluster_representative = {}

    members = set(clusters[cluster_label])
    edges = [
        (u, v, k)
        for u, v, k, d in G.edges(data=True, keys=True)
        if d.get("theme") in members
    ]
    subG = G.edge_subgraph(edges)
    if subG.number_of_nodes() > 800:
        print(f"[warn] {subG.number_of_nodes():,} nodes — pyvis may be slow/cluttered")

    out = f"{output_dir}/kg_cluster_{cluster_label}.html"
    plot_kg(subG, out)
    print(
        f"Cluster {cluster_label} | {len(members)} themes | "
        f"{subG.number_of_nodes():,} nodes | {subG.number_of_edges():,} edges | "
        f"rep: {cluster_representative.get(cluster_label)}"
    )
    return subG

In [30]:
MIN_THEMES = 3

big_clusters = sorted(
    (
        (label, len(members))
        for label, members in clusters.items()
        if len(members) >= MIN_THEMES
    ),
    key=lambda lm: -lm[1],
)

print(f"{len(big_clusters)} clusters with >= {MIN_THEMES} themes\n")
for label, size in big_clusters:
    print(f"{label:>10} : {size} themes")

971 clusters with >= 3 themes

     c2859 : 32 themes
     c2850 : 25 themes
     c3434 : 24 themes
     c2862 : 21 themes
     c2852 : 16 themes
     c3419 : 16 themes
      c671 : 15 themes
     c3439 : 15 themes
     c3423 : 14 themes
     c2855 : 14 themes
     c5454 : 13 themes
     c5145 : 11 themes
     c6955 : 11 themes
      c815 : 11 themes
     c4691 : 11 themes
     c2857 : 11 themes
      c684 : 11 themes
     c3438 : 11 themes
     c2861 : 11 themes
     c3425 : 11 themes
     c3420 : 10 themes
     c6003 : 10 themes
     c2741 : 10 themes
     c3037 : 10 themes
     c9774 : 9 themes
     c4246 : 9 themes
     c3436 : 9 themes
     c5440 : 9 themes
     c7976 : 9 themes
     c2532 : 9 themes
      c157 : 9 themes
     c2206 : 9 themes
     c4623 : 9 themes
     c9107 : 9 themes
     c2849 : 9 themes
     c3749 : 8 themes
     c7841 : 8 themes
     c2275 : 8 themes
      c326 : 8 themes
     c3424 : 8 themes
     c3431 : 8 themes
     c3527 : 8 themes
     c3429 : 8 themes

In [36]:
viz_cluster_kg("c6988")

../output/exp3_chapter6_rgcn/kg_cluster_c6988.html
Saved -> ../output/exp3_chapter6_rgcn/kg_cluster_c6988.html
Cluster c6988 | 3 themes | 11 nodes | 14 edges | rep: None
